# 04 — Inspect Beta Series Images

**Purpose:** Load and visually inspect the beta maps produced by notebook 03. By the end of this notebook you will have:
- Looked at individual trial-level beta maps to confirm they look like plausible brain activation patterns
- Averaged beta maps across trials of the same condition to get a cleaner signal
- Compared condition averages to get a first look at what differs across the four emotion conditions

This notebook is **purely diagnostic and exploratory** — it saves figures but does no statistical analysis.

---

## What are beta maps?

Recall from notebook 03: the LSS GLM produced one beta map per trial. Each beta map is a 3D brain image (same shape as your BOLD data) where the value at every voxel is the **estimated BOLD response** for that one trial — how strongly that voxel activated during that specific emotional experience.

You now have 64 such maps: 32 trials × 2 runs. They are organized by condition:

| Condition | Description | Trials per run |
|---|---|---|
| `pos_val` | Positive valence | 8 |
| `neg_val` | Negative valence | 8 |
| `pos_aro` | Positive arousal | 8 |
| `neg_aro` | Negative arousal | 8 |

**Output:** Figures saved to `Analysis/outputs/{subject}/beta_series/figures/beta_inspection/`

---
## Section 1 — Configuration and Paths

As in the previous notebooks, define everything from `subject` so paths stay general.

The beta maps from notebook 03 live inside a timestamped run directory:
```
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\runs\{run_id}\beta_maps\
```

Because the `run_id` contains a timestamp you may not remember, the safest way to find the most recent run is to **sort all run directories and take the last one**. The `Path.glob()` method finds all matching subdirectories, and Python's built-in `sorted()` puts them in alphabetical order (which equals chronological order because the timestamp is at the front):

```python
runs_dir   = beta_series_dir / "runs"
run_dirs   = sorted(runs_dir.glob("*"))   # all subdirectories, sorted by name
latest_run = run_dirs[-1]                  # last entry = most recent
beta_maps_dir = latest_run / "beta_maps"

print(f"Using run: {latest_run.name}")
print(f"Beta maps dir exists: {beta_maps_dir.exists()}")
```

Also create a folder to save inspection figures:
```python
figures_dir = beta_series_dir / "figures" / "beta_inspection"
figures_dir.mkdir(parents=True, exist_ok=True)
```

**TODO:**
- [ ] Set `subject` and `tasks`
- [ ] Define `beta_series_dir` and `beta_maps_dir` using the pattern above
- [ ] Print `latest_run.name` to confirm which run you're working with
- [ ] Create `figures_dir`

In [10]:
from pathlib import Path

subject = "sub-097"
tasks   = ["modulate1", "modulate2"]

# TODO: define project_dir and beta_series_dir
project_dir = Path(r"C:\ManzaRotation")
beta_series_dir = project_dir / "Analysis" / "outputs" / f"{subject}" / "beta_series" 
runs_dir = beta_series_dir / "runs"
# TODO: find the most recent run directory using sorted() + glob("*")
runs_sorted_dir = sorted(runs_dir.glob("*"))
latest_run = runs_sorted_dir[-1]
# TODO: define beta_maps_dir = latest_run / "beta_maps"
beta_maps_dir = latest_run / "beta_maps"
# TODO: define and create figures_dir
figures_dir = latest_run / "figures" / "beta_inspection"
figures_dir.mkdir(parents=True, exist_ok=True)
# TODO: print latest_run.name to confirm
print(latest_run.name)

20260617-224637__tR-2s__smooth-6mm__conf-6


---
## Section 2 — Discover and Catalog the Beta Maps

Before loading any images, get a clear view of what files you have. Use `Path.glob()` to find all `.nii.gz` files and parse the filenames to extract the task and trial label.

A filename looks like this:
```
sub-097_modulate1_pos_val_1_beta.nii.gz
```

You can split on `_` to extract the parts you care about. The condition name is everything between the task name and the trial number. One reliable approach: strip the known prefix (`sub-097_`) and suffix (`_beta.nii.gz`), then split the remainder:

```python
# Example: parse one filename
fname = "sub-097_modulate1_pos_val_1_beta.nii.gz"
stem  = fname.replace(f"{subject}_", "").replace("_beta.nii.gz", "")
# stem is now: "modulate1_pos_val_1"
parts = stem.split("_")   # ["modulate1", "pos", "val", "1"]
task      = parts[0]           # "modulate1"
trial_num = parts[-1]          # "1"
condition = "_".join(parts[1:-1])  # "pos_val"
```

Build a pandas DataFrame with one row per beta map, storing columns for `task`, `condition`, `trial_num`, and `path`. This catalog will make it easy to filter and group maps later.

```python
import pandas as pd

rows = []
for path in sorted(beta_maps_dir.glob("*.nii.gz")):
    stem      = path.name.replace(f"{subject}_", "").replace("_beta.nii.gz", "")
    parts     = stem.split("_")
    task      = parts[0]
    trial_num = parts[-1]
    condition = "_".join(parts[1:-1])
    rows.append({"task": task, "condition": condition, "trial_num": int(trial_num), "path": path})

catalog = pd.DataFrame(rows)
print(catalog.shape)
print(catalog.groupby(["task", "condition"]).size())
```

**TODO:**
- [ ] Build the `catalog` DataFrame using the loop above
- [ ] Print its shape — should be (64, 4)
- [ ] Print the groupby count to confirm 8 trials per condition per run
- [ ] Use `display(catalog.head(10))` to inspect a sample

In [39]:
import pandas as pd

# TODO: loop over beta_maps_dir.glob("*.nii.gz"), parse each filename,
#       build a list of dicts, convert to a DataFrame called catalog
rows = []

for path in sorted(beta_maps_dir.glob("*.nii.gz")):
    stem = path.name.replace(f"{subject}_", "").replace("_beta.nii.gz", "")
    parts = stem.split("_")   # ["modulate1", "pos", "val", "1"]
    task      = parts[0]           # "modulate1"
    trial_num = parts[-1]          # "1"
    condition = "_".join(parts[1:-1])
    rows.append({"subject": subject, "task": task, "condition": condition, "trial_num": int(trial_num), "path": path})
    
# TODO: print catalog.shape
catalog = pd.DataFrame(rows)

# TODO: print catalog.groupby(["task", "condition"]).size()
print(catalog.groupby(["task", "condition"]).size())
# TODO: display(catalog.head(20))
catalog.head(20)

task       condition
modulate1  neg_aro      8
           neg_val      8
           pos_aro      8
           pos_val      8
modulate2  neg_aro      8
           neg_val      8
           pos_aro      8
           pos_val      8
dtype: int64


,subject,task,condition,trial_num,path
0,sub-097,modulate1,neg_aro,1,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
1,sub-097,modulate1,neg_aro,2,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
2,sub-097,modulate1,neg_aro,3,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
3,sub-097,modulate1,neg_aro,4,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
4,sub-097,modulate1,neg_aro,5,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
5,sub-097,modulate1,neg_aro,6,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
6,sub-097,modulate1,neg_aro,7,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
7,sub-097,modulate1,neg_aro,8,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
8,sub-097,modulate1,neg_val,1,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
9,sub-097,modulate1,neg_val,2,C:\ManzaRotation\Analysis\outputs\sub-097\beta...


---
## Section 3 — Inspect Individual Beta Maps

Now load and plot a handful of individual trial beta maps to confirm they look like plausible brain activation patterns.

### What function to use: `plot_stat_map`

`plot_stat_map` from `nilearn.plotting` is the standard function for displaying statistical or parameter-estimate brain maps. Key arguments:

| Argument | What it does |
|---|---|
| `stat_map_img` | The 3D image to display (your beta map) |
| `threshold` | Values below this are shown as transparent. Try `0.5` to start |
| `title` | Text label on the figure |
| `colorbar` | Set to `True` to show the color scale |
| `display_mode` | Which planes to show — `'ortho'` (default), `'z'`, `'x'`, `'y'` |
| `cut_coords` | Where to slice — e.g., `(0, 0, 0)` for the MNI origin |
| `output_file` | Save the figure directly to a file path |

Loading and plotting one beta map looks like this:

```python
import nibabel as nib
from nilearn.plotting import plot_stat_map, show

# Pick one map from the catalog
example_path = catalog.loc[0, "path"]
example_img  = nib.load(example_path)

plot_stat_map(
    example_img,
    threshold=0.5,
    title=f"{catalog.loc[0, 'condition']} trial {catalog.loc[0, 'trial_num']}",
    colorbar=True,
)
show()
```

### What to look for
- Smooth, spatially coherent blobs of activation — not random speckling
- Values in a plausible range (roughly −3 to +3 for a 6mm-smoothed beta)
- Coverage across the whole brain — not just one slice or region
- No obvious artefacts like only the brain edges lighting up

Individual trial beta maps are **noisy** — don't be alarmed if a single trial looks messy. That is expected. The average across trials (Section 4) will look much cleaner.

**TODO:**
- [ ] Import `nibabel` and `plot_stat_map`
- [ ] Load and plot one map from modulate1 and one from modulate2
- [ ] Try at least two different conditions (e.g., one `pos_val` and one `neg_aro`)
- [ ] Save the figures with `output_file=` pointing to `figures_dir`

In [112]:
threshold = 0.5

for idx in range(len(catalog)):
    row = catalog.iloc[idx]

    filename = f"{task} idx-{idx}_{row['condition']}_trial-{row['trial_num']}_glass.png"
    output_path = figures_dir / filename

    trial_x_image = nib.load(row["path"])

    plot_stat_map(
        trial_x_image,
        threshold=threshold,
        vmax=1.0,
        title=f"{row['condition']} trial {row['trial_num']} idx={idx} threshold={threshold}",
        colorbar=True,
        output_file=output_path
    )

---
## Section 4 — Plot a Glass Brain View

Another useful visualization is the **glass brain**, which projects activation onto a transparent 3D brain outline so you can see the full spatial extent of activation at once.

`plot_glass_brain` works almost identically to `plot_stat_map`:

```python
from nilearn.plotting import plot_glass_brain

plot_glass_brain(
    example_img,
    threshold=0.5,
    title="Glass brain — pos_val trial 1",
    colorbar=True,
    plot_abs=False,   # set to False to show both positive and negative values
)
show()
```

Setting `plot_abs=False` is important when looking at beta maps because you want to see both positive (activation) and negative (deactivation) values, not just the absolute magnitude.

**TODO:**
- [ ] Import `plot_glass_brain`
- [ ] Plot the same example image used in Section 3 as a glass brain
- [ ] Compare the two views — what information does each one give you that the other doesn't?
- [ ] Save the figure to `figures_dir`

In [98]:
from nilearn.plotting import plot_glass_brain

plot_glass_brain(
    trial_x_image,      # pass the loaded image, same as plot_stat_map
    threshold=0.5,
    # remove vmax=1.0 entirely, or set it to match your data range:
    # vmax=5.0
    title=f"{catalog.loc[idx, 'condition']} trial {catalog.loc[idx, 'trial_num']}",
    colorbar=True,
    plot_abs=False,
    output_file = figures_dir
)
show()

---
## Section 5 — Average Beta Maps Within Each Condition

Individual trial beta maps are noisy because they are based on a single ~10-second event. Averaging across all 8 trials of the same condition (16 across both runs) gives a much cleaner picture of the typical response for that condition.

### The two-step approach

**Step 1: `concat_imgs`** — stack multiple 3D images into a single 4D image along the time axis

**Step 2: `mean_img`** — average that 4D image along the time axis, giving you back a 3D average

This is exactly the same `mean_img` function used in notebook 02 to average the BOLD timeseries — here you're just applying it to a stack of beta maps instead of BOLD volumes.

```python
from nilearn.image import concat_imgs, mean_img

# Filter the catalog to get only pos_val rows
pos_val_rows = catalog[catalog["condition"] == "pos_val"]

# Load all the images for this condition as a list
pos_val_imgs = [nib.load(p) for p in pos_val_rows["path"]]

# Stack them into a 4D image
pos_val_4d   = concat_imgs(pos_val_imgs)
print(pos_val_4d.shape)   # (x, y, z, n_trials) — should have 16 in the last dimension

# Average across the last dimension
pos_val_mean = mean_img(pos_val_4d)
print(pos_val_mean.shape)  # (x, y, z) — back to 3D
```

Repeat this for all four conditions and store the averages in a dictionary so you can easily plot and compare them:

```python
conditions = ["pos_val", "neg_val", "pos_aro", "neg_aro"]
condition_means = {}   # will hold one mean image per condition

for cond in conditions:
    rows = catalog[catalog["condition"] == cond]
    imgs = [nib.load(p) for p in rows["path"]]
    condition_means[cond] = mean_img(concat_imgs(imgs))
    print(f"{cond}: averaged {len(imgs)} maps")
```

**TODO:**
- [ ] Import `concat_imgs` and `mean_img` from `nilearn.image`
- [ ] Write the loop above to build `condition_means`
- [ ] Print the shape of each mean image to confirm they are all 3D

In [ ]:
from nilearn.image import concat_imgs, mean_img

conditions = ["pos_val", "neg_val", "pos_aro", "neg_aro"]

# TODO: loop over conditions, filter catalog, load images,
#       concat_imgs + mean_img, store result in condition_means dict

# TODO: print the shape of each mean image


(97, 115, 97, 16)
(97, 115, 97)
pos_val: averaged 16 maps
neg_val: averaged 16 maps
pos_aro: averaged 16 maps
neg_aro: averaged 16 maps


---
## Section 6 — Visualize Condition-Averaged Beta Maps

Now plot each of the four condition averages side by side so you can start to see how the brain response differs across conditions.

### Option A — Plot each condition separately

The simplest approach: loop over `condition_means` and call `plot_stat_map` once per condition:

```python
for cond, mean_map in condition_means.items():
    plot_stat_map(
        mean_map,
        threshold=0.3,          # lower threshold for averaged maps — they're less noisy
        title=f"Mean beta — {cond}",
        colorbar=True,
        display_mode="z",       # axial slices
        cut_coords=5,           # show 5 evenly spaced axial slices
        output_file=figures_dir / f"{subject}_mean_beta_{cond}.png",
    )
    show()
```

Note: `cut_coords=5` with `display_mode="z"` automatically picks 5 evenly spaced axial slices. You can also pass a specific list of z-coordinates, e.g., `cut_coords=[-20, -10, 0, 10, 20, 30, 40]`.

### Option B — Use matplotlib to put all four in a grid

For a side-by-side comparison, you can pass a matplotlib `Axes` object to `plot_stat_map` using the `axes` argument. This is more advanced — only attempt it if Option A is already working:

```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (cond, mean_map) in zip(axes.flat, condition_means.items()):
    display = plot_stat_map(
        mean_map,
        threshold=0.3,
        title=cond,
        colorbar=True,
        display_mode="z",
        cut_coords=3,
        axes=ax,
    )

plt.tight_layout()
plt.savefig(figures_dir / f"{subject}_all_condition_means.png", dpi=150)
show()
```

**TODO:**
- [ ] Start with Option A — loop over `condition_means` and plot + save each one
- [ ] Look at the four plots: which regions activate for which conditions?
- [ ] Try lowering `threshold` to `0.1` — how does the picture change?
- [ ] (Optional) Try Option B to put all four in a single figure

In [ ]:
# TODO: loop over condition_means.items() and call plot_stat_map for each condition
#       save each figure to figures_dir

# TODO (optional): try the 2x2 matplotlib grid version

---
## Section 7 — Compare Conditions: Compute a Difference Map

Averaging within conditions shows you the typical response. Subtracting two condition averages shows you what is **different** between them — which regions respond more strongly to one condition than the other.

### `math_img` — image arithmetic

`nilearn.image.math_img` lets you perform voxel-wise arithmetic on brain images using a string expression. Think of it like a calculator that operates on every voxel simultaneously:

```python
from nilearn.image import math_img

# Subtract neg_val from pos_val at every voxel
valence_contrast = math_img(
    "img1 - img2",
    img1=condition_means["pos_val"],
    img2=condition_means["neg_val"],
)
```

The result is a 3D image where:
- **Positive values** = voxels that responded more to `pos_val` than `neg_val`
- **Negative values** = voxels that responded more to `neg_val` than `pos_val`
- **Zero** = no difference

This is called a **contrast** — a basic building block of fMRI analysis.

Two scientifically interesting contrasts for this dataset:
1. **Valence contrast**: `pos_val − neg_val` — what differs between positive and negative valence?
2. **Arousal contrast**: `pos_aro − neg_aro` — what differs between positive and negative arousal?

**TODO:**
- [ ] Import `math_img` from `nilearn.image`
- [ ] Compute the valence contrast: `pos_val − neg_val`
- [ ] Compute the arousal contrast: `pos_aro − neg_aro`
- [ ] Plot both contrasts with `plot_stat_map`, using a symmetric colormap like `'RdBu_r'`
- [ ] Save both figures to `figures_dir`

In [ ]:
from nilearn.image import math_img

# TODO: compute valence_contrast = pos_val mean − neg_val mean

# TODO: compute arousal_contrast = pos_aro mean − neg_aro mean

# TODO: plot both contrasts with plot_stat_map
#       hint: pass cmap='RdBu_r' for a red-blue colormap that centers on zero

# TODO: save both figures to figures_dir

---
## Section 8 — Compare Individual Runs

So far the catalog includes maps from both runs pooled together. It is worth checking whether the two runs look consistent with each other — if one run has very different patterns, something may have gone wrong (e.g., a motion artefact).

Compute a condition mean **separately for each run** and then compare them.

To filter the catalog by both condition and task:

```python
# Get pos_val maps from modulate1 only
rows = catalog[(catalog["condition"] == "pos_val") & (catalog["task"] == "modulate1")]
```

Build a nested dictionary `run_means[task][condition]` to organize everything:

```python
run_means = {}

for task in tasks:
    run_means[task] = {}
    for cond in conditions:
        rows = catalog[(catalog["condition"] == cond) & (catalog["task"] == task)]
        imgs = [nib.load(p) for p in rows["path"]]
        run_means[task][cond] = mean_img(concat_imgs(imgs))
    print(f"{task}: done")
```

Then plot `run_means["modulate1"]["pos_val"]` and `run_means["modulate2"]["pos_val"]` side by side and look for consistency.

**TODO:**
- [ ] Build `run_means` using the nested loop above
- [ ] Pick one condition (e.g., `pos_val`) and plot the per-run means side by side
- [ ] Do the two runs look similar or different? Write your observations in the markdown cell below

In [ ]:
# TODO: build run_means nested dict

# TODO: plot modulate1 vs modulate2 for one condition

# TODO: save figures to figures_dir

*Your observations on run-to-run consistency:*

> (write here before moving on)

---
## Section 9 — Check the Distribution of Beta Values

Plots show you the spatial structure of the beta maps, but it is also useful to look at the **distribution of values across voxels** — a histogram tells you whether the values are centered near zero, whether there are outliers, and whether the scale is reasonable.

### Getting voxel values from a NIfTI image

A NIfTI image loaded with `nibabel` is a Python object. To get the raw numbers as a NumPy array, use `.get_fdata()`:

```python
import numpy as np
import matplotlib.pyplot as plt

# Get the voxel data as a 3D NumPy array
data = condition_means["pos_val"].get_fdata()
print(data.shape)   # (x, y, z)

# Flatten to 1D to plot a histogram of all voxel values
flat = data.flatten()

# Remove zeros (outside-brain voxels are stored as 0 by the mask)
flat_nonzero = flat[flat != 0]

plt.figure(figsize=(7, 4))
plt.hist(flat_nonzero, bins=80, color="steelblue", edgecolor="none")
plt.axvline(0, color="black", linewidth=1, linestyle="--")
plt.xlabel("Beta value")
plt.ylabel("Voxel count")
plt.title("Distribution of beta values — pos_val mean")
plt.tight_layout()
plt.savefig(figures_dir / f"{subject}_beta_histogram_pos_val.png", dpi=150)
plt.show()
```

**What to look for:**
- The distribution should be roughly centered near 0
- It should be roughly bell-shaped (normal-ish), not heavily skewed
- A few large positive or negative outliers are expected — these are the strongly responding voxels

**TODO:**
- [ ] Import `numpy` and `matplotlib.pyplot`
- [ ] Call `.get_fdata()` on one condition mean, flatten, and remove zeros
- [ ] Plot a histogram
- [ ] Repeat for all four conditions and compare — do any look unusual?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# TODO: call .get_fdata() on one condition mean image
# TODO: flatten to 1D and remove zeros
# TODO: plot a histogram

# TODO: loop over all four conditions and overlay or subplot their histograms

---
## Section 10 — Summary

**What you did in this notebook:**
1. Found the beta maps from the most recent LSS run
2. Built a catalog DataFrame to organize 64 maps by task, condition, and trial number
3. Inspected individual trial beta maps with `plot_stat_map` and `plot_glass_brain`
4. Averaged maps within conditions using `concat_imgs` + `mean_img`
5. Computed condition difference maps with `math_img`
6. Checked run-to-run consistency by computing per-run averages
7. Examined the distribution of voxel values with histograms

**What comes next:**

The beta maps are the raw material for beta-series **functional connectivity** analysis. The next step is to:
- Define a seed region of interest (ROI)
- Extract the mean beta value from that ROI for every trial
- Correlate that trial-by-trial timeseries with every other voxel in the brain

That correlation map is the beta-series connectivity map — it tells you which regions track trial-by-trial fluctuations in your seed region.

**TODO:**
- [ ] Print a final summary of all figures saved to `figures_dir`
- [ ] Write any observations about which conditions look most different from each other

In [ ]:
# TODO: list all files saved to figures_dir and print them
for f in sorted(figures_dir.glob("*.png")):
    print(f.name)